# ⚡ Manga Studio Cloud v0.2.0 Beta (Google Colab GPU)
### Execute o Manga Studio v0.2.0 Beta gratuitamente na nuvem da Google!
Todo o processamento pesado de imagens, OCR e IA roda nos servidores da Google com **0% de uso do seu PC**.
Equipado com **Dupla de IAs Cooperativas**: Agente 1 (Mapeamento Espacial & Typesetting) + Agente 2 (Contextualização Literária & Tradução).

---
### 🚀 Como usar:
1. No menu superior do Colab, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução** e certifique-se de que está em **T4 GPU**.
2. Configure as opções abaixo (salvar no Google Drive e ativar IA Local na GPU se desejar).
3. Clique no botão de Play `[ ▶ ]` na célula abaixo.
4. Em cerca de 30 a 60 segundos, o link seguro da Cloudflare (ex: `https://xxxx.trycloudflare.com`) será exibido.
5. Abra o link no navegador para usar o **Manga Studio completo** no computador ou no celular!

In [ ]:
# @title ▶ Iniciar Manga Studio na Nuvem (Clique no Play)
# @markdown **Deseja salvar e carregar mangás traduzidos direto do seu Google Drive?**
salvar_no_google_drive = True # @param {type:"boolean"}

# @markdown **Deseja ativar IA local na GPU (Ollama + Qwen 2.5) sem limites de cotas? (Demora ~1-2 min no primeiro uso)**
ativar_ia_local_gpu = False # @param {type:"boolean"}

import os, subprocess, time, re, sys, shutil

# 1. Garante que o diretório inicial seja sempre /content
%cd /content

# 2. Conectar Google Drive se marcado
if salvar_no_google_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs('/content/drive/MyDrive/MANGAS [PT-BR]', exist_ok=True)
        os.makedirs('/content/drive/MyDrive/MANGAS', exist_ok=True)
        os.makedirs('/content/drive/MyDrive/.manga_studio', exist_ok=True)
        print("📁 Google Drive conectado com sucesso! Obras salvas em: Google Drive > MANGAS [PT-BR]")
    except Exception as e:
        print(f"[!] Aviso Google Drive: {e}")

# 3. Baixar ou atualizar o Manga Studio
print("Atualizando / Baixando o Manga Studio do GitHub...")
if os.path.exists('/content/manga-translator/.git'):
    %cd /content/manga-translator
    !git fetch origin main > /dev/null 2>&1 || true
    !git reset --hard origin/main > /dev/null 2>&1 || true
    !git pull
else:
    %cd /content
    !rm -rf /content/manga-translator
    !git clone https://github.com/Jos9011/SKILL-TRADUCAO-MANGA.git /content/manga-translator
    %cd /content/manga-translator

try:
    from version import __version_name__
    app_version_name = __version_name__
except Exception:
    app_version_name = "v0.2.0 Beta"
print(f"📦 Versão instalada: {app_version_name}")

# 3.1 Restaura perfil e chaves do Google Drive se existirem
drive_cfg = '/content/drive/MyDrive/.manga_studio/config.json'
if os.path.exists(drive_cfg):
    try:
        shutil.copy(drive_cfg, '/content/manga-translator/config.json')
        print("🔑 Perfil e Chaves de IA restauradas automaticamente do seu Google Drive!")
    except Exception as e:
        pass

print("Verificando dependências de OCR e tradução...")
!pip install -q flask pillow opencv-python-headless rapidocr_onnxruntime deep_translator pyphen wordninja tqdm requests

# 4. Configurar IA Local na GPU T4 (Ollama) se ativado
if ativar_ia_local_gpu:
    print("\n⚡ Configurando IA Local na GPU T4 (Ollama)...")
    if not shutil.which('ollama') and not os.path.exists('/usr/local/bin/ollama'):
        print("Baixando e instalando o motor Ollama...")
        !curl -fsSL https://ollama.com/install.sh | sh
    
    if '/usr/local/bin' not in os.environ.get('PATH', ''):
        os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
        
    ollama_bin = shutil.which('ollama') or ('/usr/local/bin/ollama' if os.path.exists('/usr/local/bin/ollama') else None)
    
    if ollama_bin:
        if os.path.exists('/content/drive/MyDrive'):
            os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/.ollama_models'
            os.makedirs('/content/drive/MyDrive/.ollama_models', exist_ok=True)
            print("💾 Modelos da IA vinculados ao Google Drive para carregamento rápido.")
            
        !pkill -f 'ollama serve' || true
        time.sleep(1)
        try:
            subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            time.sleep(3)
            print("Carregando modelo Qwen 2.5 na VRAM da GPU...")
            !ollama pull qwen2.5:7b || ollama pull qwen2.5:3b
            print("✅ IA Local na GPU ativa! Sistema híbrido pronto.")
        except Exception as e:
            print(f"[!] Aviso ao iniciar Ollama: {e}. Continuando com Groq Cloud...")
    else:
        print("[!] Aviso: Ollama não encontrado. O Manga Studio continuará utilizando a nuvem (Groq/OpenRouter) com máxima velocidade.")

print("Configurando túnel seguro Cloudflare...")
!which cloudflared > /dev/null 2>&1 || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1)

cloudflared_bin = shutil.which('cloudflared') or '/usr/local/bin/cloudflared' or '/usr/bin/cloudflared'

# Encerra qualquer processo anterior para liberar a porta 5050
!pkill -f 'studio/server.py' || true
!pkill -f 'cloudflared tunnel' || true
time.sleep(1)

# Inicia o servidor Flask em segundo plano
p_srv = subprocess.Popen([sys.executable, 'studio/server.py'])
time.sleep(3)

# Inicia túnel Cloudflare para gerar o link público
p_tunnel = subprocess.Popen([cloudflared_bin, 'tunnel', '--url', 'http://127.0.0.1:5050'], stderr=subprocess.PIPE, text=True)

print('\n' + '='*60)
print(f'   🚀 MANGA STUDIO {app_version_name} INICIANDO NA NUVEM DA GOOGLE...')
print('='*60)

for line in p_tunnel.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print('\n' + '='*60)
        print(f'   ✅ MANGA STUDIO {app_version_name} PRONTO E OPERACIONAL!')
        print(f'   👉 LINK DE ACESSO: {url}')
        print('='*60)
        print('\n(Você pode abrir este link no computador ou celular.')
        print('DEIXE ESTA ABA DO COLAB ABERTA ENQUANTO ESTIVER TRADUZINDO)\n')
        break

# Thread contínua para evitar travamento do buffer de logs da Cloudflare
import threading
def _drain_stderr(pipe):
    try:
        for _ in pipe:
            pass
    except Exception:
        pass
threading.Thread(target=_drain_stderr, args=(p_tunnel.stderr,), daemon=True).start()

# Mantém a execução ativa no Google Colab monitorando o servidor
try:
    while True:
        if p_srv.poll() is not None:
            print('[!] Aviso: Servidor Manga Studio foi interrompido.')
            break
        if p_tunnel.poll() is not None:
            print('[!] Aviso: Túnel Cloudflare foi encerrado.')
            break
        time.sleep(2)
except KeyboardInterrupt:
    print('\nEncerrando Manga Studio...')
    p_srv.terminate()
    p_tunnel.terminate()


In [ ]:
# @title 🤖 (Opcional) Testar IA Local do Colab pelo Terminal
# Execute esta célula para testar se a IA local está respondendo na GPU do Colab
!ollama run qwen2.5:7b "Diga 'IA Local na GPU operacional e pronta para traduzir mangás!' em português."
